In [40]:
# Physics-Informed Machine Learning for Aftershock Forecasting

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, brier_score_loss,
                             precision_recall_curve, auc, f1_score,
                             roc_curve)
from sklearn.calibration import calibration_curve # Moved from sklearn.metrics
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import BallTree
import xgboost as xgb
import lightgbm as lgb
import shap
import warnings
warnings.filterwarnings('ignore')

# High-quality plotting settings
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.linewidth': 1.5,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'figure.figsize': (8, 6)
})

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment ready. High-resolution figures will be generated.")

Environment ready. High-resolution figures will be generated.


In [41]:
# Simulation parameters (balanced, same as before)
T_MAX_DAYS = 5 * 365
AREA_KM2 = 500 * 500
MAG_COMPLETE = 3.0
MAG_MAX = 8.0
B_VALUE = 1.0
MU = 2.0 / AREA_KM2
K0 = 0.2  # Fixed after adaptive search
ALPHA = 1.0
C = 0.01
P = 1.2
SIGMA = 5.0
BETA_DEPTH = 0.2

def generate_catalog(K0_val, max_events=50000):
    # Same as before – generate ETAS catalog
    n_background = np.random.poisson(MU * AREA_KM2 * T_MAX_DAYS)
    bg_times = np.random.uniform(0, T_MAX_DAYS, n_background)
    bg_x = np.random.uniform(0, 500, n_background)
    bg_y = np.random.uniform(0, 500, n_background)
    bg_depth = np.random.uniform(0, 30, n_background)
    bg_mag = np.zeros(n_background)
    for i in range(n_background):
        while True:
            m = MAG_COMPLETE + np.random.exponential(scale=1.0/B_VALUE)
            if m <= MAG_MAX:
                bg_mag[i] = m
                break
    catalog = pd.DataFrame({'time': bg_times, 'x': bg_x, 'y': bg_y,
                            'depth': bg_depth, 'mag': bg_mag, 'parent': -1})
    events_list = [catalog]
    idx = 0
    while idx < len(events_list[0]) and len(events_list[0]) < max_events:
        ev = events_list[0].iloc[idx]
        mu_direct = K0_val * np.exp(ALPHA * (ev['mag'] - MAG_COMPLETE))
        n_direct = np.random.poisson(mu_direct)
        for _ in range(n_direct):
            t_delay = C * (np.random.pareto(P) - 1)
            t_off = ev['time'] + t_delay
            if t_off > T_MAX_DAYS:
                continue
            r = np.random.normal(0, SIGMA)
            angle = np.random.uniform(0, 2*np.pi)
            x_off = ev['x'] + r * np.cos(angle)
            y_off = ev['y'] + r * np.sin(angle)
            if x_off < 0 or x_off > 500 or y_off < 0 or y_off > 500:
                continue
            d_off = ev['depth'] + np.random.normal(0, 2)
            d_off = np.clip(d_off, 0, 30)
            while True:
                m_off = MAG_COMPLETE + np.random.exponential(scale=1.0/B_VALUE)
                if m_off <= MAG_MAX: break
            depth_factor = np.exp(-BETA_DEPTH * ev['depth'])
            if np.random.rand() > depth_factor: continue
            events_list[0] = pd.concat([events_list[0], pd.DataFrame([{
                'time': t_off, 'x': x_off, 'y': y_off,
                'depth': d_off, 'mag': m_off, 'parent': idx
            }])], ignore_index=True)
        idx += 1
        if len(events_list[0]) >= max_events: break
    return events_list[0].sort_values('time').reset_index(drop=True)

catalog = generate_catalog(K0)
print(f"Generated {len(catalog)} events")

Generated 5047 events


In [42]:
def boussinesq_index(mag, depth, d0=1.0):
    moment = 10**(1.5 * mag + 9.1)
    return np.log10(moment / ((depth + d0)**2))

def coulomb_proxy(mag, depth, friction=0.6):
    # Simplified Coulomb stress change proxy:
    # ΔCFS ∝ (shear stress increase) - μ * (normal stress increase)
    # Approximate shear stress by magnitude, normal stress by depth
    shear = 10**(1.5*mag + 9.1) / (depth + 1)  # rough proxy
    normal = -shear * friction  # negative sign for clamping
    return shear + normal

catalog['stress_index'] = boussinesq_index(catalog['mag'], catalog['depth'])
catalog['coulomb_proxy'] = coulomb_proxy(catalog['mag'], catalog['depth'])

In [43]:
def label_larger_follower(df, time_window_days=7, distance_km=50, mag_increase=1.0):
    n = len(df)
    labels = np.zeros(n, dtype=int)
    t = df['time'].values
    coords = df[['x', 'y']].values
    tree = BallTree(coords, metric='euclidean')
    mags = df['mag'].values
    for i in range(n):
        future_mask = (t > t[i]) & (t <= t[i] + time_window_days)
        future_idx = np.where(future_mask)[0]
        if len(future_idx) == 0: continue
        nearby = tree.query_radius(coords[i].reshape(1,-1), r=distance_km)[0]
        candidates = np.intersect1d(future_idx, nearby)
        if len(candidates) == 0: continue
        if np.any(mags[candidates] >= mags[i] + mag_increase):
            labels[i] = 1
    return labels

catalog['target'] = label_larger_follower(catalog)
print(f"Positive fraction: {catalog['target'].mean():.4f}")

split_time = catalog['time'].quantile(0.8)
train_df = catalog[catalog['time'] <= split_time].copy()
test_df = catalog[catalog['time'] > split_time].copy()

Positive fraction: 0.2217


In [44]:
features = ['mag', 'depth', 'stress_index', 'coulomb_proxy']
X_train = train_df[features].values
y_train = train_df['target'].values
X_test = test_df[features].values
y_test = test_df['target'].values

scaler = RobustScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline (only mag+depth)
X_train_base = X_train_scaled[:, :2]
X_test_base = X_test_scaled[:, :2]

models = {
    'RF (baseline)': RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
    'RF + phys': RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
    'GB + phys': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_SEED),
    'XGB + phys': xgb.XGBClassifier(n_estimators=100, random_state=RANDOM_SEED),
    'LGB + phys': lgb.LGBMClassifier(n_estimators=100, random_state=RANDOM_SEED),
    'LR + phys': LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)
}

# Train models
results = {}
models['RF (baseline)'].fit(X_train_base, y_train)
for name, model in models.items():
    if name == 'RF (baseline)': continue
    model.fit(X_train_scaled, y_train)

[LightGBM] [Info] Number of positive: 832, number of negative: 3205
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000254 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 4037, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.206094 -> initscore=-1.348635
[LightGBM] [Info] Start training from score -1.348635


In [45]:
def evaluate_model(model, X, y, name):
    if name == 'RF (baseline)':
        y_proba = model.predict_proba(X)[:,1]
    else:
        y_proba = model.predict_proba(X)[:,1]
    roc_auc_value = roc_auc_score(y, y_proba) # Renamed to avoid conflict
    brier = brier_score_loss(y, y_proba)
    precision, recall, _ = precision_recall_curve(y, y_proba)
    pr_auc = auc(recall, precision)
    f1 = f1_score(y, (y_proba>=0.5).astype(int), average='weighted')
    return {'AUC': roc_auc_value, 'Brier': brier, 'PR-AUC': pr_auc, 'F1': f1, 'proba': y_proba}

for name, model in models.items():
    if name == 'RF (baseline)':
        res = evaluate_model(model, X_test_base, y_test, name)
    else:
        res = evaluate_model(model, X_test_scaled, y_test, name)
    results[name] = res

# Create results table
results_df = pd.DataFrame(results).T
print(results_df.round(4))

                    AUC     Brier    PR-AUC        F1  \
RF (baseline)  0.752293  0.176288  0.564354  0.721256   
RF + phys      0.752618   0.17824  0.563141  0.707341   
GB + phys      0.810837  0.158179  0.644383  0.719711   
XGB + phys     0.772666  0.170883  0.569498  0.713048   
LGB + phys     0.786929  0.164153  0.603771  0.725474   
LR + phys      0.818492  0.156576  0.650415  0.725195   

                                                           proba  
RF (baseline)  [0.13, 0.63, 0.01, 0.0, 0.01, 0.19, 0.16, 0.11...  
RF + phys      [0.12, 0.59, 0.01, 0.0, 0.0, 0.09, 0.4, 0.11, ...  
GB + phys      [0.1739782545911338, 0.28159810191837087, 0.06...  
XGB + phys     [0.15691836, 0.2567471, 0.0357562, 0.03261224,...  
LGB + phys     [0.2596053697966821, 0.23089749007490842, 0.07...  
LR + phys      [0.16387443986725037, 0.24423453284982044, 0.0...  


In [ ]:
# Figure 1: ROC Curves (high resolution)
plt.figure(figsize=(8,6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={res['AUC']:.3f})")
plt.plot([0,1], [0,1], 'k--', lw=1, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves – Aftershock Forecasting', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('Fig2_ROC_highres.png', dpi=300)
plt.show()

# Figure 2: PR Curves
plt.figure(figsize=(8,6))
for name, res in results.items():
    precision, recall, _ = precision_recall_curve(y_test, res['proba'])
    plt.plot(recall, precision, lw=2, label=f"{name} (PR-AUC={res['PR-AUC']:.3f})")
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves', fontsize=14)
plt.legend(loc='lower left', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('Fig3_PR_highres.png', dpi=300)
plt.show()

In [ ]:
best_model = models['LR + phys']
explainer = shap.LinearExplainer(best_model, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)
shap.summary_plot(shap_values, X_test_scaled, feature_names=features,
                  show=False, plot_size=(10,4))
plt.title('SHAP Feature Importance – Logistic Regression', fontsize=14)
plt.tight_layout()
plt.savefig('Fig4_SHAP_highres.png', dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
for name, res in results.items():
    fop, mpv = calibration_curve(y_test, res['proba'], n_bins=10)
    plt.plot(mpv, fop, marker='s', lw=2, label=name)
plt.plot([0,1], [0,1], 'k--', lw=1, label='Perfect Calibration')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Frequency')
plt.title('Calibration Curves – Aftershock Forecasting')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('Fig1_Calibration_highres.png', dpi=300)
plt.show()